# CipherMark — évaluation sur Google Colab

Exécute la chaîne de tests CipherMark : tests unitaires, chaîne crypto bout-à-bout,
et le protocole de robustesse du hash perceptuel avec **DINOv2** (le run décisif).

Exécuter les cellules dans l'ordre. GPU non obligatoire, mais accélère DINOv2
(`Exécution > Modifier le type d'exécution > T4 GPU`).

In [ ]:
# 1) Cloner le depot (branche crystal)
# Si le depot est prive : decommenter la ligne TOKEN et coller un token GitHub
# (Settings > Developer settings > Personal access tokens, portee 'repo').
REPO = "github.com/ngueagho/distseal-meta.git"
# import os; os.environ['GH_TOKEN'] = 'ghp_votre_token_ici'
import os
url = f"https://{os.environ['GH_TOKEN']}@{REPO}" if 'GH_TOKEN' in os.environ else f"https://{REPO}"
!git clone --branch crystal --depth 1 {url} distseal-meta
%cd distseal-meta

In [ ]:
# 2) Dependances (tout le reste est preinstalle sur Colab)
!pip install -q reedsolo

In [ ]:
# 3) Tests unitaires (attendu : 14/14 OK)
!PYTHONPATH=. python -m tests.ciphermark.run_all

In [ ]:
# 4) Chaine crypto bout-a-bout
#    (attendu : round-trip 20/20, replay 0/20, faux positifs 0/50)
!python -m scripts.ciphermark.gen_keys --out-dir ./keys
!PYTHONPATH=. python -m scripts.ciphermark.eval_ciphermark --n 20 --keys-dir ./keys

In [ ]:
# 5) Corpus a l'echelle de l'article DistSeal : ~50 000 scenes
#    (le papier evalue FID et robustesse sur 50 000 images generees)
#    Source principale : COCO unlabeled2017, telechargement ~19 Go --
#    rapide sur Colab (~5-10 min), converti en 256x256 au fil de l'eau.
#    + kodak24 et bsds300 pour la continuite avec les runs precedents.
#    Prevoir ~25-40 min au total et ~5 Go de disque pour le corpus final.
!PYTHONPATH=. python -m scripts.ciphermark.build_corpus --out corpus-test \
    --kodak --bsds --coco unlabeled2017 --coco-max 50000
import os; print(len(os.listdir("corpus-test")), "images")

In [ ]:
# 6) LE RUN DECISIF : robustesse du canal hash avec DINOv2, capacite 16 (defaut)
#    ~50k images x 16 distorsions : compter 2-4 h sur T4, ~1 h sur A100.
#    Le traitement est par morceaux (--chunk) : la memoire reste bornee et
#    la progression s'affiche. En cas de session courte, reduire --n.
#    Lecture : colonne 'recuperable' = fraction d'images dont le hash reste
#    corrigeable. C'est elle qui decide de la viabilite.
!mkdir -p results
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 60000 --data-dir ./corpus-test --device cuda --chunk 512 \
    --csv results/phash_dino_rs32.csv

In [ ]:
# 6bis) Comparaison : meme protocole avec le fallback DCT (borne basse)
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 60000 --data-dir ./corpus-test --device cuda --chunk 512 \
    --backbone dct --csv results/phash_dct_reelles.csv

In [ ]:
# 7) Contre-verification a capacite reduite (rs-nsym 16 = capacite 8 octets),
#    pour le tableau comparatif du chapitre 3. Le defaut du script est
#    desormais rs-nsym 32 (capacite 16), justifie par la fenetre de
#    discrimination mesuree.
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 60000 --data-dir ./corpus-test --device cuda --chunk 512 \
    --rs-nsym 16 --csv results/phash_dino_rs16.csv

In [ ]:
# 8) Recuperer les CSV (pour le chapitre 3 du memoire)
from google.colab import files
for f in ["results/phash_dino_rs32.csv", "results/phash_dct_reelles.csv", "results/phash_dino_rs16.csv"]:
    try:
        files.download(f)
    except Exception as e:
        print(f"{f}: {e}")